In [1]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=32c12e390f8a0624a5b81ce84c869b95ae03d1caa1b74f118502038d71eded51
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [2]:
from datasets import load_dataset
import textwrap
import pandas as pd
import numpy as np
import plotly.express as px
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq

In [3]:
data_set = load_dataset("abisee/cnn_dailymail", name="3.0.0")
df = pd.DataFrame(data_set['train'][:10000])
print("Done and the Shape is:", df.shape)

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Done and the Shape is: (10000, 3)


In [4]:
df

,article,highlights,id
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,ee8871b15c50d0db17b0179a6d2beab35065f1e9
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...",06352019a19ae31e527f37f7571c6dd7f0c5da37
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",24521a2abb2e1f5e34e6824e0f9e56904a2b0e88
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a
...,...,...,...
9995,"(Oprah.com) -- As a trained chef, restaurant o...",Chef Laura Pensiero: Don't let diet hijack you...,75cd2144a44f902e14b887bf927a08ad6b5fbf54
9996,"Islamabad, Pakistan (CNN) -- The Pakistani mil...",NEW: Taliban source says commander alive and w...,734ca9a754dd444ed50362375e256fd72d82def5
9997,(CNN) -- Bastian Schweinsteiger rescued a poin...,Midfielder Bastian Schweinsteiger rescues a po...,8ff4e932c84ed30759427a128879f0af1c8a156a
9998,"(CNN) -- Haiti's earthquake is creating ""a gen...","Experts say Haiti's earthquake is creating ""a ...",12e6c95b6f47092e953523ae556cb6391a7021f6


In [5]:
print("Article:\n")
print(textwrap.fill(df['article'].iloc[0]))
print("\n" + "="*100 + "\n")
print("Highlights:\n")
print(df['highlights'].iloc[0])

Article:

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains
access to a reported £20 million ($41.1 million) fortune as he turns
18 on Monday, but he insists the money won't cast a spell on him.
Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the
Phoenix" To the disappointment of gossip columnists around the world,
the young actor says he has no plans to fritter his cash away on fast
cars, drink and celebrity parties. "I don't plan to be one of those
people who, as soon as they turn 18, suddenly buy themselves a massive
sports car collection or something similar," he told an Australian
interviewer earlier this month. "I don't think I'll be particularly
extravagant. "The things I like buying are things that cost about 10
pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to
gamble in a casino, buy a drink in a pub or see the horror film
"Hostel: Part II," currently six places below his number one movie on
the UK box office chart. Deta

In [6]:
df.drop(columns=['id'],inplace=True)

In [7]:
print(df.info())
print("\n" + "="*100)

print(f'the duplicate is: ' ,df.duplicated().sum())
print("\n" + "="*100)

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   article     10000 non-null  object
 1   highlights  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB
None

the duplicate is:  867



,0
article,0
highlights,0


In [8]:
df = df.drop_duplicates(subset=['article']).reset_index(drop=True)
df

,article,highlights
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa..."
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non..."
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical..."
...,...,...
9127,"(Oprah.com) -- As a trained chef, restaurant o...",Chef Laura Pensiero: Don't let diet hijack you...
9128,"Islamabad, Pakistan (CNN) -- The Pakistani mil...",NEW: Taliban source says commander alive and w...
9129,(CNN) -- Bastian Schweinsteiger rescued a poin...,Midfielder Bastian Schweinsteiger rescues a po...
9130,"(CNN) -- Haiti's earthquake is creating ""a gen...","Experts say Haiti's earthquake is creating ""a ..."


In [9]:
print(f'The Dublicated is :',df.duplicated().sum())

The Dublicated is : 0


In [10]:
df.describe()

,article,highlights
count,9132,9132
unique,9132,9088
top,(SuperYachtWorld) -- They may not be big on lo...,Rick Steves writes European travel guidebooks ...
freq,1,5


In [11]:
df['article_length'] = df['article'].apply(lambda x: len(x.split()))
df['highlights_length'] = df['highlights'].apply(lambda x: len(x.split()))

display(df[['article_length','highlights_length']].describe())

,article_length,highlights_length
count,9132.000000,9132.000000
mean,623.026500,44.711345
std,309.013442,8.084830
min,18.000000,11.000000
25%,376.000000,38.000000
50%,579.000000,45.000000
75%,829.000000,51.000000
max,1831.000000,85.000000


In [12]:
px.histogram(df, x='article_length',
             nbins=30,
             title = 'Distribution of Article Length',
             color_discrete_sequence=['blue'])

In [13]:
px.box(df,
       y = ['article_length','highlights_length'],
       title = 'Box Plot for Article_length Highlights_length',
       color_discrete_sequence=['pink'],
       )

In [14]:
fig_violin = px.violin(df,
                       y = ['article_length', 'highlights_length'],
                       title = 'Violin Plot',
                       box = True,
                       points = 'outliers',
                       color_discrete_sequence = ['Violet'])
fig_violin.show()

In [15]:
px.density_heatmap(df,
                   x = 'article_length',
                   y = 'highlights_length',
                   title = 'Heatmap of Article Length vs. Highlights Length',
                   color_continuous_scale='plasma')

In [16]:
df_clean = df[
    (df['article_length'] >= 100) & (df['article_length'] <= 1000) &
    (df['highlights_length'] >= 10) & (df['highlights_length'] <= 100)
    ].copy().reset_index(drop=True)
df_clean

,article,highlights,article_length,highlights_length
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,455,41
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,698,49
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...",743,43
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",414,27
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",973,44
...,...,...,...,...
7952,Spin bowlers Harbhajan Singh and Murali Kartik...,India have recalled Harbhajan Singh and Murali...,324,42
7953,(CNN) -- Self-help guru James Ray was arrested...,Self-help guru James Ray was arrested Wednesda...,309,56
7954,(OPRAH.com) -- Allow me to introduce myself. I...,Columnist Lisa Kogan says please don't say the...,943,56
7955,"Islamabad, Pakistan (CNN) -- The Pakistani mil...",NEW: Taliban source says commander alive and w...,224,49


In [17]:
df_train, df_test = train_test_split(df_clean, test_size = 0.2, random_state = 42)

print("train size (Train):", df_train.shape)
print("test size (Test):", df_test.shape)

train size (Train): (6365, 4)
test size (Test): (1592, 4)


In [18]:
model = AutoModelForSeq2SeqLM.from_pretrained('facebook/bart-large-cnn')
tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-cnn')

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [19]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

def preprocess_function(examples):
    model_inputs = tokenizer(examples['article'],
                             max_length = 1024,
                             truncation = True,
                             padding = 'max_length')

    labels = tokenizer(examples['highlights'],
                       max_length = 128,
                       padding = 'max_length',
                       truncation = True)

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/6365 [00:00<?, ? examples/s]

Map:   0%|          | 0/1592 [00:00<?, ? examples/s]

In [20]:
rouge_metric = evaluate.load('rouge')

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  result = rouge_metric.compute(predictions = decoded_preds, references = decoded_labels, use_stemmer = True)
  return {k: round(v * 100, 4) for k, v in result.items()}


training_args = Seq2SeqTrainingArguments(
    output_dir = "./bart_summarization_results",
    eval_strategy = "epoch",
    learning_rate = 2e-5,
    num_train_epochs = 1,
    weight_decay = 0.01,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    predict_with_generate = True,
    )

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    )

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.547134,0.536072,41.735800,19.820300,29.581900,39.289200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1592, training_loss=0.62100848959918, metrics={'train_runtime': 6348.5739, 'train_samples_per_second': 1.003, 'train_steps_per_second': 0.251, 'total_flos': 1.379362079244288e+16, 'train_loss': 0.62100848959918, 'epoch': 1.0})

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
save_path = "/content/drive/MyDrive/my_saved_bart_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/my_saved_bart_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_saved_bart_model/tokenizer.json')